# Projekt 01 (basic) — Ein N-Gramm-Sprachmodell

**Modul 08 — NLP 1**

Du baust ein statistisches **Sprachmodell** auf echtem Text (*The Adventures of
Sherlock Holmes*): N-Gramm-Zaehlungen, **Add-k-Glaettung**, **Interpolation**,
Bewertung per **Perplexitaet** und **Textgenerierung** durch Sampling (Skript Teil 1).

Du erlebst direkt: warum ungeglaettete Modelle scheitern, warum hoehere N-Gramme
kleinere Perplexitaet haben (bis die Daten zu duenn werden) und warum Mischen
(Interpolation) am besten ist.

## Setup
Nur Standardbibliothek. Die erste Zelle **laedt den Korpus** von Project Gutenberg
(~600 KB) nach `datasets/` und cached ihn (die Datei wird per `.gitignore` nicht
eingecheckt). Kernel des Repo-`.venv` waehlen und Zellen der Reihe nach ausfuehren;
dann Aufgabe 1–3 loesen.

## Teil A — Daten & Vorverarbeitung (vorgegeben)
Korpus laden, in Saetze und Tokens zerlegen, Train/Test splitten, Vokabular mit `<unk>`/`<s>`/`</s>` bauen und N-Gramme zaehlen.

In [ ]:
# ---- Korpus laden (echter Text: "The Adventures of Sherlock Holmes") -------
import os, re, urllib.request, math, random
from collections import Counter, defaultdict

DATA_DIR = "daten"
os.makedirs(DATA_DIR, exist_ok=True)
CORPUS = os.path.join(DATA_DIR, "sherlock.txt")
URL = "https://www.gutenberg.org/files/1661/1661-0.txt"

if not os.path.exists(CORPUS):
    print("Lade Korpus von Project Gutenberg ...")
    req = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
    raw = urllib.request.urlopen(req).read().decode("utf-8", errors="ignore")
    # Gutenberg-Rahmentext (Lizenz etc.) entfernen
    start = raw.find("*** START OF")
    end = raw.find("*** END OF")
    if start != -1 and end != -1:
        raw = raw[raw.find("\n", start) + 1:end]
    with open(CORPUS, "w", encoding="utf-8") as f:
        f.write(raw)
    print("gespeichert:", CORPUS)

text = open(CORPUS, encoding="utf-8").read()
print(f"Korpuslaenge: {len(text):,} Zeichen")

In [ ]:
# ---- Tokenisierung & Satzsegmentierung -------------------------------------
def sentences(text):
    """Sehr einfache Segmentierung: Saetze an .!? trennen, dann Woerter."""
    text = text.replace("\n", " ")
    for raw in re.split(r"[.!?]+", text):
        toks = re.findall(r"[a-z']+", raw.lower())
        if toks:
            yield toks

sents = list(sentences(text))
random.seed(42); random.shuffle(sents)
split = int(0.9 * len(sents))
train_sents, test_sents = sents[:split], sents[split:]
print(f"{len(sents):,} Saetze  ->  {len(train_sents):,} train / {len(test_sents):,} test")
print("Beispiel:", train_sents[0][:12])

In [ ]:
# ---- Vokabular mit <unk>, <s>, </s> ----------------------------------------
# Woerter, die im Training nur EINMAL vorkommen, werden zu <unk> -> das Modell
# lernt eine Verteilung fuer unbekannte Woerter, und die Perplexitaet bleibt endlich.
BOS, EOS, UNK = "<s>", "</s>", "<unk>"

train_counts = Counter(w for s in train_sents for w in s)
vocab = {w for w, c in train_counts.items() if c >= 2}
vocab |= {BOS, EOS, UNK}
print(f"Vokabulargroesse |V| = {len(vocab):,}")

def normalize(sent):
    return [BOS] + [w if w in vocab else UNK for w in sent] + [EOS]

train = [normalize(s) for s in train_sents]
test = [normalize(s) for s in test_sents]

In [ ]:
# ---- N-Gramm-Zaehlungen (Unigramm, Bigramm, Trigramm) ----------------------
uni = Counter()
bi = defaultdict(Counter)     # bi[w1][w2] = C(w1,w2)
tri = defaultdict(Counter)    # tri[(w1,w2)][w3] = C(w1,w2,w3)

for s in train:
    for i, w in enumerate(s):
        uni[w] += 1
        if i >= 1:
            bi[s[i-1]][w] += 1
        if i >= 2:
            tri[(s[i-2], s[i-1])][w] += 1

N_uni = sum(uni.values())
V = len(vocab)
print(f"Tokens gesamt N = {N_uni:,},  |V| = {V:,}")
print("haeufigste Woerter:", uni.most_common(8))

### Aufgabe 1 — Add-k-Wahrscheinlichkeiten & Perplexitaet
Implementiere die geglaetteten Wahrscheinlichkeiten (1a) und die Perplexitaet (1b). Vergleiche Uni-/Bi-/Trigramm.

In [ ]:
# ---- AUFGABE 1a: Add-k-geglaettete Wahrscheinlichkeiten --------------------
# Add-k-Formel (Skript 1.3):  P(w | kontext) = (C(kontext,w) + k) / (C(kontext) + k*|V|)
# Nutze uni, bi, tri und N_uni, V.
def p_unigram(w, k=1.0):
    # TODO
    raise NotImplementedError

def p_bigram(w, w1, k=1.0):
    # TODO:  (bi[w1][w] + k) / (uni[w1] + k*V)
    raise NotImplementedError

def p_trigram(w, w1, w2, k=1.0):
    # TODO:  Kontext ist (w2, w1);  Nenner = sum(tri[(w2,w1)].values()) + k*V
    raise NotImplementedError

print("P(holmes | mr) =", round(p_bigram("holmes", "mr"), 5))

In [ ]:
# ---- AUFGABE 1b: Perplexitaet ----------------------------------------------
# PP = exp( -1/N * sum_i log P(w_i | kontext) ),  summiert ueber alle Positionen i>=1
# aller Testsaetze;  N = Anzahl dieser Positionen.
def perplexity(sentences, prob_fn):
    # prob_fn(sent, i) liefert P(w_i | vorherige Woerter)
    # TODO
    raise NotImplementedError

pp_uni = perplexity(test, lambda s, i: p_unigram(s[i]))
pp_bi  = perplexity(test, lambda s, i: p_bigram(s[i], s[i-1]))
pp_tri = perplexity(test, lambda s, i: p_trigram(s[i], s[i-1], s[i-2]) if i >= 2
                                        else p_bigram(s[i], s[i-1]))
print(f"Perplexitaet (Add-1):  Unigramm {pp_uni:7.1f} | Bigramm {pp_bi:7.1f} | Trigramm {pp_tri:7.1f}")

### Aufgabe 2 — Interpolation
Mische die drei Ordnungen. Mit kleinem $k$ sollte die Perplexitaet deutlich unter die der Einzelmodelle fallen.

In [ ]:
# ---- AUFGABE 2: Interpolation ----------------------------------------------
# P_interp = l1*P_uni + l2*P_bi + l3*P_tri  (mit l1+l2+l3 = 1).
# Nutze ein KLEINERES k (z.B. 0.01) — Add-1 ist fuer grosse |V| zu grob.
# Am Satzanfang (i<2) gibt es keinen Trigramm-Kontext: nimm dort das Bigramm.
def p_interp(s, i, l1=0.1, l2=0.3, l3=0.6, k=0.01):
    # TODO
    raise NotImplementedError

pp_interp = perplexity(test, p_interp)
print(f"Perplexitaet (Interpolation): {pp_interp:7.1f}")

### Aufgabe 3 — Textgenerierung
Sample Saetze aus dem Bigramm-Modell. Der Text klingt „holmes-haft“, aber grammatisch grob — genau die Grenze von N-Grammen.

In [ ]:
# ---- AUFGABE 3: Textgenerierung durch Sampling -----------------------------
# Starte bei <s>. Ziehe wiederholt das naechste Wort aus dem Bigramm-Modell
# bi[letztes_wort] (Woerter gewichtet nach ihren Zaehlungen — random.choices).
# Stoppe bei </s> oder max_len. Gib den Satz ohne <s>/</s> zurueck.
def generate(max_len=25, seed=0):
    rng = random.Random(seed)
    # TODO
    raise NotImplementedError

for seed in range(5):
    print(" •", generate(seed=seed))

## Reflexion (kurz, schriftlich)
1. Ueberraschung: mit **Add-1** ist das Trigramm *schlechter* als das Unigramm.
   Warum? (Ueber-Glaettung: bei grossem $|V|$ frisst $k\cdot|V|$ im Nenner die
   ganze Masse.) Und warum behebt Interpolation mit kleinem $k$ das?
2. Was passiert mit der Perplexitaet, wenn du in `perplexity` ein ungeglaettetes
   Modell nutzt und ein Testwort im Training nie in diesem Kontext vorkam?
3. Warum verbessert Interpolation gegenueber dem reinen Trigramm? (Skript: Rueckfall
   auf niedrigere Ordnung bei duennen Daten.)
4. Der generierte Text ist lokal plausibel, global aber Unsinn. Welche
   Sprach-Eigenschaft koennen N-Gramme prinzipiell nicht erfassen? (Ausblick Modul 09.)

Musterantworten am Ende der Loesung im Ordner `solution/`.